# Language Model and Sequence Generation

This lab aims to familiarize you with the concepts of language model and sequence generation.

From a corpus of texts written by Voltaire, we'll train a recurrent model based on character sequences.

## Checking GPU Usage

Go to the `Runtime > Change runtime type` menu and check that we're on Python 3 and that the hardware accelerator is set to "GPU".

In [ ]:
!nvidia-smi

## Fetching the Data

The various Voltaire texts making up the corpus were obtained from gutenberg.org. The headers, footers and prefaces were removed beforehand: we wouldn't want our language model to learn to write gutenberg.org's disclaimers or the publisher's prefaces.

We'll only use the `voltaire/voltaire_clean.txt` file.

In [ ]:
!rm -rf voltaire
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s voltaire
print("─" * 50)
!ls -l voltaire/
print("─" * 50)
!cat voltaire/voltaire_clean.txt


In [ ]:
# Use this cell to explore the data a bit.
!wc voltaire/voltaire_clean.txt

In [ ]:
# Uncomment to download the file
# import google.colab
# google.colab.files.download('voltaire/voltaire_clean.txt')

## Installing and Importing PyTorch Lightning and the Other Required Libraries

In [ ]:
!pip install -q lightning torchmetrics

In [ ]:
import pathlib

import lightning
import numpy
import torch
import torchmetrics
from lightning.pytorch.callbacks import Callback
from lightning.pytorch.loggers import CSVLogger
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

## Loading the Data and Extracting the Vocabulary

- *Load the file's text in lowercase into the `text` variable*
- *Extract all the characters used in `text` into the `index_to_char` list, sorted alphabetically. This list is used to go from a character's numerical representation to the character itself*
- *Build a dictionary `char_to_index` to go from a character to its numerical representation*

In [ ]:
text = "toto"
index_to_char = ["o", "t"]
char_to_index = {"o": 0, "t": 1}

### Solution

In [ ]:
text = pathlib.Path("voltaire/voltaire_clean.txt").read_text().lower()
print(f"{text[:100]}…")
print(f"Number of characters: {len(text)}")

In [ ]:
index_to_char = sorted(set(text))
print(index_to_char)
print(f"Dictionary size: {len(index_to_char)}")

In [ ]:
char_to_index = {c: i for i, c in enumerate(index_to_char)}
print(char_to_index)

## Preprocessing

Split the text into sequences of 30 characters, every 3 characters. The targets will be the 31st characters

In [ ]:
maxlen = 30
step = 3
sentences = []
next_chars = []
for i in range(0, len(text) - maxlen, step):
  sentences.append(text[i: i + maxlen])
  next_chars.append(text[i + maxlen])
print(f"Number of sequences: {len(sentences)}")

In [ ]:
print(sentences[0])

## Creating the Input and Output Arrays

*Create the `X` and `y` variables containing the encoded input and output data.*

In [ ]:
# Your code here

### Solution

In [ ]:
def encode(sentence: str) -> torch.Tensor:
  return torch.tensor([char_to_index[char] for char in sentence])


X = torch.stack([encode(sentence) for sentence in sentences])
y = encode(next_chars)

print(X.shape, y.shape)

## Preparing the Model

The model below embeds each character, runs the sequence through a recurrent network (an LSTM by default; the `cell` parameter allows a GRU or a simple RNN instead), then scores every possible character from the network's last output.

The goal is to minimize the loss: the lower it is, the closer the model is to Voltaire's language model.

In [ ]:
class LanguageModel(lightning.LightningModule):
  """Lightning wrapper: predict the character following a sequence."""

  def __init__(self,
               model: nn.Module,
               learning_rate: float = 1e-3,
               sequence_len: int = maxlen,
               num_classes: int = len(index_to_char)) -> None:
    super().__init__()
    self.save_hyperparameters(ignore=["model"])
    self.model = model
    # An example input lets Lightning display the shapes of the input and output
    # tensors of each layer in the model summary
    self.example_input_array = torch.zeros(1, sequence_len, dtype=torch.long)
    # torchmetrics needs one metric instance per stage
    self.accuracies = nn.ModuleDict({
        f"{stage}_accuracy": torchmetrics.Accuracy(task="multiclass",
                                                   num_classes=num_classes)
        for stage in ("train", "val")
    })

  def forward(self, sentences: torch.Tensor) -> torch.Tensor:
    return self.model(sentences)

  def _step(self, batch: tuple[torch.Tensor, torch.Tensor],
            stage: str) -> torch.Tensor:
    sentences, next_chars = batch
    logits = self(sentences)
    loss = nn.functional.cross_entropy(logits, next_chars)
    accuracy = self.accuracies[f"{stage}_accuracy"]
    accuracy(logits, next_chars)
    self.log(f"{stage}_loss", loss, on_step=False, on_epoch=True,
             prog_bar=True)
    self.log(f"{stage}_accuracy", accuracy, on_step=False, on_epoch=True,
             prog_bar=True)
    return loss

  def training_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                    batch_index: int) -> torch.Tensor:
    return self._step(batch, "train")

  def validation_step(self, batch: tuple[torch.Tensor, torch.Tensor],
                      batch_index: int) -> torch.Tensor:
    return self._step(batch, "val")

  def configure_optimizers(self) -> torch.optim.Optimizer:
    return torch.optim.Adam(self.parameters(),
                            lr=self.hparams.learning_rate)

In [ ]:
class CharRNN(nn.Module):
  """Embed the characters, run a recurrent network, score the next one."""

  def __init__(self,
               embedding_dim: int = 8,
               hidden_dim: int = 128,
               num_layers: int = 1,
               cell: type[nn.RNNBase] = nn.LSTM) -> None:
    super().__init__()
    self.embedding = nn.Embedding(len(index_to_char), embedding_dim)
    self.rnn = cell(embedding_dim,
                    hidden_dim,
                    num_layers=num_layers,
                    batch_first=True)
    self.head = nn.Linear(hidden_dim, len(index_to_char))

  def forward(self, sentences: torch.Tensor) -> torch.Tensor:
    embedded = self.embedding(sentences)
    outputs, _ = self.rnn(embedded)
    # Only the output of the last timestep matters: it's the summary
    # of the whole sequence
    return self.head(outputs[:, -1])


def build_model(embedding_dim: int = 8,
                hidden_dim: int = 128,
                learning_rate: float = 1e-3
                ) -> lightning.LightningModule:
  return LanguageModel(CharRNN(embedding_dim, hidden_dim),
                       learning_rate=learning_rate)

## Text Generation

The following callback runs every `every` epochs, a parameter that can be given at initialization.

It generates characters one by one, progressively shifting the given input so that it's always `maxlen` characters long.

[`torch.multinomial`](https://docs.pytorch.org/docs/stable/generated/torch.multinomial.html) samples from a probability distribution, so it's our main tool to exploit the network's output: the softmax applied to the logits indeed defines such a distribution.

In [ ]:
class SamplingCallback(Callback):
  def __init__(self, n_steps: int = 200, every: int = 10) -> None:
    super().__init__()
    self.n_steps = n_steps
    self.every = every

  @torch.no_grad()
  def on_train_epoch_end(self,
                         trainer: lightning.Trainer,
                         pl_module: lightning.LightningModule) -> None:
    epoch = trainer.current_epoch + 1
    if epoch % self.every == 0:
      print()
      print("─" * 50)
      print(f"After {epoch} epochs:")
      start_index = numpy.random.randint(0, len(text) - maxlen - 1)
      sentence = text[start_index:start_index + maxlen]
      print("─" * 50)
      print(f"Generating from: \"{sentence}\"")
      print("─" * 50)
      print(sentence, end="")

      pl_module.eval()
      for _ in range(self.n_steps):
        logits = pl_module(encode(sentence)[None, :].to(pl_module.device))
        word_probas = logits.softmax(dim=-1)
        next_index = int(torch.multinomial(word_probas, 1))
        next_char = index_to_char[next_index]
        sentence = sentence[1:] + next_char
        print(next_char, end="")
      pl_module.train()
      print()

## Training

We keep the end of the corpus (10% of the sequences) as a validation set, to track the loss on texts the model hasn't seen. The split isn't random: two neighboring sequences share 27 characters, so a random split would put almost the same sequences on both sides.

In [ ]:
batch_size = 4096
n_train = int(X.shape[0] * 0.9)
train_loader = DataLoader(TensorDataset(X[:n_train], y[:n_train]),
                          batch_size=batch_size,
                          shuffle=True)
validation_loader = DataLoader(TensorDataset(X[n_train:], y[n_train:]),
                               batch_size=batch_size)

model = build_model()
trainer = lightning.Trainer(max_epochs=100,
                            accelerator="auto",
                            devices=1,
                            logger=CSVLogger("logs", name="char_rnn"),
                            enable_checkpointing=False,
                            callbacks=[SamplingCallback(n_steps=200,
                                                        every=10)])
trainer.fit(model, train_loader, validation_loader)

## A Solution With Several LSTM Layers

In [ ]:
def build_deep_model(embedding_dim: int = 8,
                     hidden_dim: int = 128,
                     learning_rate: float = 1e-3
                     ) -> lightning.LightningModule:
  return LanguageModel(CharRNN(embedding_dim, hidden_dim, num_layers=2),
                       learning_rate=learning_rate)

In [ ]:
deep_model = build_deep_model()
deep_trainer = lightning.Trainer(
    max_epochs=100,
    accelerator="auto",
    devices=1,
    logger=CSVLogger("logs", name="deep_char_rnn"),
    enable_checkpointing=False,
    callbacks=[SamplingCallback(n_steps=200, every=10)])
deep_trainer.fit(deep_model, train_loader, validation_loader)

## Comparing Architectures

Try different recurrent networks ([`LSTM`](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html), [`GRU`](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html), [`RNN`](https://docs.pytorch.org/docs/stable/generated/torch.nn.RNN.html)), of different sizes, with `CharRNN`'s `cell` parameter. Give each configuration a few epochs before judging it, or use [Optuna](https://optuna.org/) to explore automatically.

Compare them on the validation loss (`val_loss`): which cell does best? Does the validation loss end up rising?

In [ ]:
# Your code here

### Solution

In [ ]:
def train_cell(cell: type[nn.RNNBase], name: str,
               max_epochs: int = 20) -> float:
  model = LanguageModel(CharRNN(cell=cell))
  trainer = lightning.Trainer(max_epochs=max_epochs,
                              accelerator="auto",
                              devices=1,
                              logger=CSVLogger("logs", name=name),
                              enable_checkpointing=False)
  trainer.fit(model, train_loader, validation_loader)
  # Validation loss of the last epoch
  return float(trainer.callback_metrics["val_loss"])


val_losses = {name: train_cell(cell, name)
              for name, cell in [("lstm", nn.LSTM), ("gru", nn.GRU),
                                 ("rnn", nn.RNN)]}
for name, val_loss in val_losses.items():
  print(f"{name}: validation loss {val_loss:.3f}")

Usually, the LSTM and the GRU reach similar losses, and the simple RNN does worse: it struggles to keep information over 30 time steps (vanishing gradient). A bigger model (`hidden_dim`, `num_layers`) lowers the training loss, but ends up overfitting: its validation loss rises again, and that's the time to stop training (Lightning's [`EarlyStopping`](https://lightning.ai/docs/pytorch/stable/common/early_stopping.html) callback).